# Lab 6 — Speech features CTC and recognition error analysis

COMP-672 · John Glossner · 2 points

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/glossner/COMP-672-Labs/blob/main/labs/week06_starter.ipynb)

**First: File → Save a copy in Drive.** Sign in to Google to run and save. Rename your copy, for example `COMP672_Lab06_YourName.ipynb`, and work in that copy. No GitHub account is needed for this path.

Opening this notebook does not execute code, save your answers to GitHub, or share your work. Use the cell play buttons yourself. The functions marked `NotImplementedError` are exercises for you to complete. The supplied checks and examples are scaffolding, not an exhaustive grader.

Use a **CPU** runtime. All data are original synthetic instructional examples generated by the code below; no datasets, models, credentials, Drive mount, or repository clone are required.

[Course lab index and submission instructions](https://github.com/glossner/COMP-672-Labs/blob/main/README.md) · [Handout](https://github.com/glossner/COMP-672-Labs/blob/main/labs/LAB06.md)


## Install the lab environment

Use Colab’s **default Python CPU runtime**; you do not need to select an older Python runtime. This revision supports Python **3.11–3.13**, including the Python 3.13 runtime in Google’s current backend snapshot. Run this cell before the lab imports. It prints your actual Python version and installs only missing or mismatched pinned packages from PyPI (and the official CPU PyTorch index for Labs 3 and 5). Binary wheels are required so setup will not try to compile scientific packages.

If Colab prompts for a restart after installation, select **Runtime → Restart session**, then rerun from this cell. Do not change the runtime’s system Python with `apt`, `conda`, or `update-alternatives`. A fresh runtime is preferable if you already installed unrelated packages.

Nothing is installed or run until you execute a cell. Files in the runtime are temporary; save your notebook and download your results before disconnecting.

**Updating an existing Drive copy:** GitHub changes do not update your copy automatically. Keep your current copy as a backup. Reopen this lab from the course README and save a new copy in Drive, then transfer your work; or replace only the two setup/environment code cells and these setup instructions in your existing copy with the matching lab’s latest cells. Restart the session and rerun setup. Do not overwrite your completed exercise or analysis cells.


In [ ]:
import sys
import subprocess
import importlib.metadata as metadata

if not (3, 11) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError("This lab supports Python 3.11–3.13. Reopen the latest notebook from the course README if your runtime is newer.")

print("Runtime Python:", sys.version.split()[0])

LAB_PACKAGES = {'numpy': '2.1.3', 'scipy': '1.16.3', 'scikit-learn': '1.6.1', 'matplotlib': '3.10.0'}
def installed_version(name):
    try:
        return metadata.version(name).split("+")[0]
    except metadata.PackageNotFoundError:
        return None

needed = [f"{name}=={version}" for name, version in LAB_PACKAGES.items()
          if name != "torch" and installed_version(name) != version]
if needed:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--only-binary=:all:", *needed])
print("Package setup complete. Restart the session if Colab requests it.")


In [ ]:
import platform
import importlib

LAB_VERSIONS = {"python": platform.python_version()}
for distribution, module_name in [("numpy", "numpy"), ("scipy", "scipy"),
                                  ("scikit-learn", "sklearn"), ("matplotlib", "matplotlib")]:
    module = importlib.import_module(module_name)
    LAB_VERSIONS[distribution] = module.__version__
for name, expected in LAB_PACKAGES.items():
    if LAB_VERSIONS[name].split("+")[0] != expected:
        raise RuntimeError(f"Restart the session: {name} is already imported at a different version.")
print(LAB_VERSIONS)
print("CPU environment ready. Synthetic data will be generated by the lab functions.")


# Lab 6 — Speech features CTC and recognition error analysis

**2 points.** Chapters 15, 16; outcomes LO6, LO8. Allow 4–5 hours including the 60-minute lab meeting. Complete the Python starter or the Jupyter starter, not both. The scaffolding contains intentional `NotImplementedError` gaps.

## Implementation tasks

1. Implement no-padding framing and verify the 16-kHz one-second example yields 98 frames of 400 samples.

2. Implement windowing, power spectrum, supplied Mel-filter projection, and log flooring. Plot or tabulate a feature slice with units.

3. Implement CTC collapse and enumerate compatible paths for a tiny example using the supplied exact enumerator.

4. Implement edit alignment and return S, D, I, N, and WER using the declared tie and tokenization conventions.


Functions to complete: `frame_signal`, `log_mel`, `ctc_collapse`, `word_errors`.

## Required experiments

- Compare clean and noise-corrupted two-tone features, then compare two frame lengths while keeping the hop explicit. Show the resulting dimensions.

- Compute per-utterance and pooled WER for the supplied transcripts. Add a repeated-label CTC example and an insertion-heavy transcript whose WER exceeds one.

## Brief analysis in your notebook or code comments

1. Why are these tone experiments useful for feature debugging but insufficient evidence about phoneme recognition?

2. Why does removing blanks before collapsing repeats produce a wrong CTC transcript?

3. Why can an unweighted average of utterance WER differ from pooled corpus WER?


Answer each question in approximately 2–4 sentences beside the results. This is part of the lab, with no separate report.

## Submission and rubric

Submit the link to your completed, saved Colab copy (or edited notebook/source if working locally), a results JSON file, and an execution command. Follow the save/share steps in the repository README. For Colab, state “Run cells from top to bottom in a fresh CPU runtime.” Include versions, seed, split policy, settings, runtime, and clear labels. No points depend on a positive gain.

| Criterion | Points | Evidence |
| --- | --- | --- |
| Implementation | 0.8 | Required functions implement the intended mathematics and boundary behavior |
| Experiments | 0.6 | Both required comparisons are run and outputs are recorded |
| Comprehension | 0.4 | Brief answers accurately connect results to the three analysis questions |
| Reproducibility | 0.2 | Runnable submission with configuration, seed, and data scope |
| Total | 2.0 | Partial credit is proportional to the evidence supplied |

Split implementation credit equally among required functions; split experiment credit equally between the two comparisons. Assess the three analysis answers together on a 0–0.4 scale; award 0.4 for all three accurate and grounded, 0.2 for partial understanding, 0 for absent or fundamentally incorrect explanations. Intermediate credit is allowed.


## Starter code

Complete the exercise functions below. Keep the supplied checks and add the cases required by the handout.

In [ ]:
"""Week 6 student starter. Synthesized tones demonstrate DSP; supplied transcripts test WER."""


In [ ]:
import itertools


In [ ]:
import numpy as np


In [ ]:
def frame_signal(x,frame=400,hop=160):
    """No centering or padding. Reject signals shorter than one frame."""
    raise NotImplementedError("Complete this laboratory task.")


In [ ]:
def mel_filters(sample_rate=16000,n_fft=512,bands=40):
    hz_to_mel=lambda f:2595*np.log10(1+f/700)
    mel_to_hz=lambda m:700*(10**(m/2595)-1)
    edges=mel_to_hz(np.linspace(hz_to_mel(0),hz_to_mel(sample_rate/2),bands+2))
    bins=np.arange(n_fft//2+1)*sample_rate/n_fft
    return np.array([np.maximum(0,np.minimum((bins-a)/(b-a),(c-bins)/(c-b))) for a,b,c in zip(edges,edges[1:],edges[2:])])


In [ ]:
def log_mel(x,sample_rate=16000,frame=400,hop=160,n_fft=512,bands=40):
    raise NotImplementedError("Complete this laboratory task.")


In [ ]:
def ctc_collapse(path,blank=0):
    raise NotImplementedError("Complete this laboratory task.")


In [ ]:
def ctc_probability(frame_probs,target,blank=0):
    """Exact enumeration for small examples only: exponential in frame count."""
    total=0.
    for path in itertools.product(range(frame_probs.shape[1]),repeat=len(frame_probs)):
        if ctc_collapse(path,blank)==tuple(target):
            total+=float(np.prod([frame_probs[t,v] for t,v in enumerate(path)]))
    return total


In [ ]:
def word_errors(reference,hypothesis):
    """Whitespace tokenization; unit costs. Tie preference: match, substitution, deletion, insertion.
    Return S,D,I,N and WER. An empty reference raises to avoid an undefined denominator.
    """
    raise NotImplementedError("Complete this laboratory task.")


In [ ]:
def checks():
    assert frame_signal(np.zeros(16000)).shape==(98,400)
    assert log_mel(np.zeros(16000)).shape==(98,40)
    assert np.isfinite(log_mel(np.zeros(16000))).all()
    assert ctc_collapse([1,1,0,1,2,2,0])==(1,1,2)
    assert np.isclose(ctc_probability(np.array([[.4,.6],[.4,.6]]),[1]),.84)
    assert word_errors('we need a blue car','we need blue cars')['WER']==.4
    assert word_errors('a','a b c')['WER']==2.


In [ ]:
def experiment():
    rng=np.random.default_rng(17); t=np.arange(16000)/16000
    x=.6*np.sin(2*np.pi*220*t)+.2*np.sin(2*np.pi*440*t)
    clean=log_mel(x); noisy=log_mel(x+.15*rng.normal(size=len(x)))
    refs=['we need a blue car','the radio is ready','turn left at the gate']
    hypotheses=['we need blue cars','the radio ready','turn right at the gate']
    rows=[word_errors(r,h) for r,h in zip(refs,hypotheses)]
    return {'log_mel_shape':list(clean.shape),'feature_mean_absolute_noise_change':float(np.mean(abs(clean-noisy))),'utterance_errors':rows,'corpus_wer':sum(r['S']+r['D']+r['I'] for r in rows)/sum(r['N'] for r in rows),'ctc_p_a':ctc_probability(np.array([[.4,.6],[.4,.6]]),[1]),'note':'No ASR system was trained; acoustic signals and error cases are controlled instructional examples.'}


## Run checks and the supplied experiment

Complete the exercise functions first. A `NotImplementedError` here is expected in an untouched starter. The result dictionary covers the supplied baseline; the handout also requires comparisons and analysis that you must add below.


In [ ]:
import time

checks()
_started = time.perf_counter()
results = experiment()
baseline_runtime_seconds = time.perf_counter() - _started
results


## Your required comparisons

Add the additional experiments, checks, plots/tables, measured runtimes, and clearly labeled results required by the handout. Add their JSON-compatible summaries to `additional_results`.


In [ ]:
# TODO: implement and run the remaining required comparisons here.
additional_results = {}


## Brief analysis

1. Why are these tone experiments useful for feature debugging but insufficient evidence about phoneme recognition?

2. Why does removing blanks before collapsing repeats produce a wrong CTC transcript?

3. Why can an unweighted average of utterance WER differ from pooled corpus WER?

Write approximately 2–4 sentences for each question in the next cell, tied to your measured results.

1. _Your response:_

2. _Your response:_

3. _Your response:_


## Record configuration and export results

Complete the configuration fields and include all required comparison results. Run the export cell after your experiments. The JSON will also appear in the saved notebook output so a viewer can read it. Download the generated JSON via Colab’s **Files** sidebar (file menu → Download) and attach it with your notebook link in the course submission channel. The runtime file alone is not saved in Drive or shared with your notebook.


In [ ]:
# TODO: document the settings you actually used.
configuration = {
    "seed": None,
    "split_policy": "",
    "settings_and_changes": "",
    "additional_experiment_runtime_seconds": None,
    "execution": "Run cells from top to bottom in a fresh CPU runtime",
}


In [ ]:
import json
from pathlib import Path

def json_ready(value):
    # Support NumPy scalars/arrays and nested result dictionaries.
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if hasattr(value, "tolist"):
        return value.tolist()
    return value

payload = {
    "lab": 6,
    "versions": LAB_VERSIONS,
    "configuration": configuration,
    "baseline_runtime_seconds": baseline_runtime_seconds,
    "results": results,
    "additional_results": additional_results,
}
results_path = Path("week06_results.json")
results_json = json.dumps(json_ready(payload), indent=2, allow_nan=False)
results_path.write_text(results_json + "\n", encoding="utf-8")
print(results_json)
print("Download", results_path.name, "from the Files sidebar before disconnecting.")


## Save and submit

1. Finish the code, required comparisons, and three analysis responses. Run the completed notebook from top to bottom in a fresh CPU runtime and retain the outputs.
2. **File → Save** your Drive copy and wait for saving to finish. Ensure **Edit → Notebook settings → Omit code cell output when saving this notebook** is turned off. Reopen your copy to verify the code, analysis, and results remain visible.
3. Download `week06_results.json` from the Files sidebar. Runtime files disappear when the runtime is deleted.
4. In **your copy**, select **Share → General access → Anyone with the link → Viewer**, then **Copy link** and **Done**. Anyone holding this link can view the full saved notebook, including code, output, and comments. Keep unrelated private information out.
5. Submit **your saved copy’s link** and results JSON through the submission channel specified by John Glossner. Do not submit this blank GitHub starter’s link. Verify the link opens for a signed-out viewer. If your school account blocks link sharing, ask the instructor for an approved alternative.

These actions require you to run, save, and share deliberately; this notebook never does them automatically. Cloning or opening the repository alone does not save your answers. A GitHub fork is optional, requires a GitHub account, and is public because the source repository is public; use Drive for the default course workflow.
